# # Main
[add a description/summary/abstract here later]

## 1. Setup
Install all required libraries and/or set up the environment.

Note: these older Numpy and Torch versions are needed otherwise Recbole (v1.2.1) breaks

In [2]:
!python --version

Python 3.11.0


In [3]:
!pip list

Package                   Version
------------------------- ----------------
absl-py                   2.3.1
aiosignal                 1.4.0
annotated-types           0.7.0
asttokens                 3.0.0
attrs                     25.3.0
certifi                   2025.7.14
charset-normalizer        3.4.2
click                     8.2.1
colorama                  0.4.4
colorlog                  4.7.2
comm                      0.2.2
contourpy                 1.3.2
cycler                    0.12.1
debugpy                   1.8.15
decorator                 5.2.1
executing                 2.2.0
filelock                  3.18.0
fonttools                 4.59.0
frozenlist                1.7.0
fsspec                    2025.7.0
grpcio                    1.73.1
humanize                  4.12.3
idna                      3.10
ipykernel                 6.30.0
ipython                   9.4.0
ipython_pygments_lexers   1.1.1
jedi                      0.19.2
Jinja2                    3.1.6
joblib      

In [1]:
# Core packages
!pip install -U pip
!pip install numpy==1.26.4
!pip install pandas==2.3.1
!pip install torch==2.5.1
!pip install scikit-learn==1.7.1
!pip install matplotlib==3.10.3

# Recommender system packages
!pip install lenskit==2025.2.0
!pip install recbole==1.2.1

  Using cached lenskit-2025.2.0-py3-none-any.whl.metadata (7.8 kB)
Using cached lenskit-2025.2.0-py3-none-any.whl (227 kB)
  Attempting uninstall: lenskit
    Found existing installation: lenskit 2025.1.1
    Uninstalling lenskit-2025.1.1:
      Successfully uninstalled lenskit-2025.1.1


### 1.1 Load Drive (Google Colab)
Load Google Drive and copy the datasets folder

In [9]:
import os
import shutil
from google.colab import drive


drive.mount("/content/drive", readonly=True)
source = "/content/drive/My Drive/Workspace/Thesis/Repository/unreasonable-effectiveness-recsys/datasets"
destination = "./datasets"

shutil.copytree(source, destination)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileExistsError: [Errno 17] File exists: './datasets'

### 1.2 GPU Check
Use torch to check if GPU(s) is/are availabe in the current environment

In [27]:
import torch
import os


print("=== GPU Detection ===")
print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("\n=== GPU Details ===")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}:")
        print(f"  Name: {props.name}")
        print(f"  Memory: {props.total_memory / 1024**3:.1f} GB")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print(f"  Multi-processors: {props.multi_processor_count}")
        
        # Test GPU accessibility
        try:
            torch.cuda.set_device(i)
            test_tensor = torch.tensor([1.0]).cuda()
            print(f"  Status: ✓ Accessible")
        except Exception as e:
            print(f"  Status: ✗ Error - {e}")

    print(f"\n=== Current GPU ===")
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Current device name: {torch.cuda.get_device_name()}")
    
    print(f"\n=== Environment Variables ===")
    print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', 'Not set')}")
    
else:
    print("No CUDA GPUs detected")


=== GPU Detection ===
CUDA available: False
Number of GPUs: 0
No CUDA GPUs detected


## 2. Parameters
Global variables and shared parameters

### 2.1 Experiment variables
Fixed experiment parameters

In [5]:

RECOMMENDATIONS = 10
SEED = 42
# SIZES = [i * 0.1 for i in range(1, 11)] # 0.1, 0.2, ..., 1.0
SIZES = [0.1, 0.25, 0.5, 0.75, 1.0]
COLUMN_NAMES = {"user_id": "user_id", "item_id": "item_id", "rating": "rating"}


### 2.2 Hyperparameters
Shared hyperparameters

In [2]:

TEST_SIZE = 0.2
PARTITIONS = 5 # For cross-validation
KNN_K = 20
FEATURES = 50
TRAINING_RECOMMENDATIONS = 100


## 3. Load
Load the desired dataset as a Pandas DataFrame of user ID, item ID, and rating in that order

In [11]:
from enum import Enum
import pandas as pd


class Datasets(Enum):
  MOVIELENS = "movielens"
  GOODREADS = "goodreads"
  AMAZON = "amazon"
  MOVIETWEETINGS = "movietweetings"
  PERSONALITY = "personality"

def load(dataset: Datasets = Datasets.MOVIELENS) -> pd.DataFrame:
  path = f"./datasets/{dataset.value}/ratings.csv"
  df = pd.read_csv(path, sep=',', usecols=[0, 1, 2], names=[
    COLUMN_NAMES["user_id"],
    COLUMN_NAMES["item_id"],
    COLUMN_NAMES["rating"]
  ])
  df = df.reset_index(drop=True)
  return df


## 4. Sample
Take a sample of size N within the globally defined range

In [20]:
import pandas as pd


def sample(df: pd.DataFrame, size: float = SIZES[(len(SIZES) // 2) - 1]) -> pd.DataFrame:
  if size not in SIZES:
    raise ValueError(f"Size must be one of '{SIZES}'")

  size = int(size * len(df))
  return df.sample(size, random_state=SEED)


## 5. Splitting (unused)
Data-frame splitting

### 5.1. Horizontal split (unused)
Split the data-frame into training and testing data-frames

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


def horizontal_split(df: pd.DataFrame, test_size: float = TEST_SIZE):
  return train_test_split(df, test_size=test_size, random_state=SEED)


### 5.2. Vertical split (unused)
Split the training and testing data into X and y data-frames (features and target)

In [ ]:
import pandas as pd


def vertical_split(train: pd.DataFrame, test: pd.DataFrame):
  X_train, X_test = train.drop(columns=["rating"]), test.drop(columns=["rating"])
  y_train, y_test = train["rating"], test["rating"]
  return X_train, X_test, y_train, y_test


## 6. LensKit
Use LensKit to train and evaluate a given data-frame returning a single float of the mean NDCG

In [ ]:
from enum import Enum
import pandas as pd
from lenskit.data import from_interactions_df, UserIDKey, ItemListCollection
from lenskit.splitting import SampleFrac, crossfold_users
from lenskit.knn import ItemKNNScorer, ItemKNNConfig, UserKNNScorer, UserKNNConfig
from lenskit.als import BiasedMFScorer
from lenskit.pipeline import topn_pipeline
from lenskit.batch import recommend
from lenskit.metrics import RunAnalysis, NDCG


class Scorer(Enum):
  ITEM_KNN = 0
  BIASED_MF = 1
  USER_KNN = 2
  BIAS = 3

SCORERS = {
  Scorer.ITEM_KNN.name: lambda: ItemKNNScorer(ItemKNNConfig(
    max_nbrs=KNN_K, min_nbrs=KNN_MIN_K, feedback="explicit",
  )),
  Scorer.BIASED_MF.name: lambda: BiasedMFScorer(features=FEATURES),
  Scorer.USER_KNN.name: lambda: UserKNNScorer(UserKNNConfig(
    max_nbrs=KNN_K, min_nbrs=KNN_MIN_K, feedback="explicit",
  )),
  Scorer.BIAS.name: lambda: BiasScorer(),
}

def use_lenskit(df: pd.DataFrame, scorer: Scorer = Scorer.ITEM_KNN) -> float:
  dataset = from_interactions_df(
    df,
    user_col=COLUMN_NAMES["user_id"],
    item_col=COLUMN_NAMES["item_id"],
    rating_col=COLUMN_NAMES["rating"],
    timestamp_col=None,
  )

  get = SCORERS[scorer.name]
  model = get()
  pipe = topn_pipeline(model)

  all_test = ItemListCollection.empty(key=UserIDKey)
  all_recs = ItemListCollection.empty(["model", "user_id"])

  for split in crossfold_users(dataset, PARTITIONS, SampleFrac(TEST_SIZE)):
    all_test.add_from(split.test)
    fit = pipe.clone()
    fit.train(split.train)
    recs = recommend(fit, split.test.keys(), n=TRAINING_RECOMMENDATIONS) # type: ignore
    all_recs.add_from(recs, model=f"Only")

  ran = RunAnalysis()
  ran.add_metric(NDCG(k=RECOMMENDATIONS, gain=COLUMN_NAMES["rating"]))
  results = ran.measure(all_recs, all_test)
  result = results.list_summary().iloc[0].iloc[0]

  return float(result)


## 7. RecBole
Use RecBole to train and evaluate a given data-frame returning a single float of the mean NDCG

### 7.1. Parameters
Recbole specific parameters and shared variables

In [16]:
ROOT_DIRECTORY = "./recbole/"
DATASETS_DIRECTORY = ROOT_DIRECTORY + "datasets/"
DATASET_TEMP_NAME = "temp"
DATASET_TEMP_DIRECTORY = DATASETS_DIRECTORY + f"{DATASET_TEMP_NAME}/"
TEMP_ATOMIC_FILE_NAME = f"{DATASET_TEMP_NAME}.inter"
TEMP_ATOMIC_FILE_PATH = DATASET_TEMP_DIRECTORY + TEMP_ATOMIC_FILE_NAME

RECBOLE_CONFIG = {
  # Environment
  "seed": SEED,
  "data_path": DATASETS_DIRECTORY,
  "checkpoint_dir": ROOT_DIRECTORY + "saved/",

  # Data
  "field_separator": ',',
  "seq_separator": '\n',
  "USER_ID_FIELD": COLUMN_NAMES["user_id"],
  "ITEM_ID_FIELD": COLUMN_NAMES["item_id"],
  "RATING_FIELD": COLUMN_NAMES["rating"],
  "TIME_FIELD": None,
  "load_col": {"inter": [COLUMN_NAMES["user_id"], COLUMN_NAMES["item_id"], COLUMN_NAMES["rating"]]},

  # Training
  ##########

  # Evaluation
  "eval_args": {
    "split": {"RS": [10 - (TEST_SIZE * 10), 0, (TEST_SIZE * 10)]},
  },
  "metrics": ["NDCG"],
  "topk": RECOMMENDATIONS,
}

### 7.2. Prepare
Save the interactions data-frame as an "Atomic" (CSV-like) .inter file

In [17]:
import pandas as pd
import os


def save_as_atomic(df: pd.DataFrame) -> None:
  os.makedirs(os.path.dirname(TEMP_ATOMIC_FILE_PATH), exist_ok=True)
  with open(TEMP_ATOMIC_FILE_PATH, "w") as f:
    features = [f"{name}:float" if name == COLUMN_NAMES["rating"] else f"{name}:token" for name in df.columns]
    f.write(','.join(features) + '\n')
  df.to_csv(TEMP_ATOMIC_FILE_PATH, mode="a", index=False, header=False)

### 7.3. Run
Create the atomic file, call "run_recbole" using the config dictionary, and returns the NDCG@[RECOMMENDATIONS] as well

In [13]:
from enum import Enum
import pandas as pd
from recbole.quick_start import run_recbole


class Model(Enum):
  ITEM_KNN = "ItemKNN"  # Item K-Nearest Neighbours
  POPULARITY = "Pop"  # Popularity
  FACTORED_ITEM_SIMILARITY = "FISM"  # Factored Item Similarity Models
  NEURAL_ATTENTIVE_ITEM_SIMILARITY = "NAIS"  # Neural Attentive Item Similarity Model

def use_recbole(df: pd.DataFrame, model: Model = Model.ITEM_KNN) -> float:
  save_as_atomic(df)
  result = run_recbole(model=model.value, dataset=DATASET_TEMP_NAME, config_dict=RECBOLE_CONFIG)
  test_result = result["test_result"]
  for key, value in test_result.items():
    if "ndcg" in key.lower():
      return float(value)
  raise ValueError("No NDCG metric found")


g:\My Drive\Workspace\Thesis\Repository\unreasonable-effectiveness-recsys\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-27 16:23:25,998	INFO util.py:159 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-07-27 16:23:29,141	INFO util.py:159 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


## 8. Plotting
Graph plotting and vizualisation of results

### 8.1. Types
Type definitions for shared types

In [7]:
from typing import Dict, TypeAlias
# from pydantic import BaseModel, ValidationError

# d[library][algorithm][dataset][size] -> float
Results: TypeAlias = Dict[str, Dict[str, Dict[str, Dict[str, float]]]]

### 8.2. Prepare
Prepare and return a dictionary that would hold the results

In [18]:

# Dictionary so that 'd[library][algorithm][dataset][size] -> float' holds the NDCG
def prepare_results(sizes: list[float] = SIZES, libraries: list[str] = ["LensKit", "RecBole"]) -> Results:
  results = {
    library: {
      algorithm.name: {
        dataset.name: {
          str(int(size * 100.0)) + '%': float(-1) for size in sizes
        } for dataset in Datasets
      } for algorithm in (Scorer if library == "LensKit" else Model)
    } for library in libraries
  }
  return results


In [19]:
import yaml
import os

def save_results(results: dict) -> None:
  """Saves the results dictionary to './results/latest.yaml'."""
  directory = "./results/"
  filepath = os.path.join(directory, "latest.yaml")
  
  os.makedirs(directory, exist_ok=True)
  
  with open(filepath, "w") as f:
    yaml.dump(results, f, indent=2, sort_keys=False)
  
  print(f"Results saved to {filepath}")

def load_results(filepath: str = "./results/latest.yaml") -> dict:
  """Loads a results dictionary from a YAML file."""
  with open(filepath, "r") as f:
    return yaml.safe_load(f)

save_results(prepare_results())
load_results()

Results saved to ./results/latest.yaml


{'LensKit': {'ITEM_KNN': {'MOVIELENS': {'10%': -1.0,
    '25%': -1.0,
    '50%': -1.0,
    '75%': -1.0,
    '100%': -1.0},
   'GOODREADS': {'10%': -1.0,
    '25%': -1.0,
    '50%': -1.0,
    '75%': -1.0,
    '100%': -1.0},
   'AMAZON': {'10%': -1.0,
    '25%': -1.0,
    '50%': -1.0,
    '75%': -1.0,
    '100%': -1.0},
   'MOVIETWEETINGS': {'10%': -1.0,
    '25%': -1.0,
    '50%': -1.0,
    '75%': -1.0,
    '100%': -1.0},
   'PERSONALITY': {'10%': -1.0,
    '25%': -1.0,
    '50%': -1.0,
    '75%': -1.0,
    '100%': -1.0}},
  'BIASED_MF': {'MOVIELENS': {'10%': -1.0,
    '25%': -1.0,
    '50%': -1.0,
    '75%': -1.0,
    '100%': -1.0},
   'GOODREADS': {'10%': -1.0,
    '25%': -1.0,
    '50%': -1.0,
    '75%': -1.0,
    '100%': -1.0},
   'AMAZON': {'10%': -1.0,
    '25%': -1.0,
    '50%': -1.0,
    '75%': -1.0,
    '100%': -1.0},
   'MOVIETWEETINGS': {'10%': -1.0,
    '25%': -1.0,
    '50%': -1.0,
    '75%': -1.0,
    '100%': -1.0},
   'PERSONALITY': {'10%': -1.0,
    '25%': -1.0,
    '50%

### 8.3. Plot results
Handle a given 'Results' dictionary instance

In [13]:
import matplotlib.pyplot as plt


MARKERS = ['o', '^', '2', 's', 'P', '*', 'X', 'D', '|']
COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple",
          "tab:brown", "tab:pink", "tab:gray", "tab:olive", "tab:cyan"]

def plot_results(results: Results, figsize=(10, 12)) -> None:
  # Setup
  libraries = list(results.keys())
  fig, axes = plt.subplots(len(libraries), 1, figsize=figsize)

  # If only one library, make axes iterable
  if len(libraries) == 1:
    axes = [axes]

  for library_index, library in enumerate(libraries):
    ax = axes[library_index]

    algorithms = list(results[library].keys())
    datasets = list(results[library][algorithms[0]].keys())  # Get datasets from first algorithm

    # Plot each algorithm-dataset combination
    for algorithm_index, algorithm in enumerate(algorithms):
      for dataset_index, dataset in enumerate(datasets):
        # Extract sizes and values
        size_value = results[library][algorithm][dataset]
        sizes = list(size_value.keys())
        # sizes = [f"{int(size * 100)}%" for size in sizes]
        values = list(size_value.values())

        marker = MARKERS[algorithm_index]
        color = COLORS[dataset_index]

        ax.plot(
          sizes, values,
          color=color,
          marker=marker,
          linewidth=2,
          markersize=10,
          label=f'{algorithm.capitalize()} - {dataset.capitalize()}',
        )

        # Customize subplot
        ax.set_title(library, fontsize=14, fontweight="bold")
        ax.set_xlabel("Dataset Size", fontsize=12)
        ax.set_ylabel(f"NDCG@{RECOMMENDATIONS}", fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

        # Set x-axis to log scale if sizes vary widely
        # if max(sizes) / min(sizes) > 10:
        #     ax.set_xscale("log")

  plt.tight_layout()
  plt.show()


# Example usage with sample data
def create_sample_data():
    """Create sample data matching your structure"""
    return {
        "LensKit": {
            "ItemKNN": {
                "MovieLens": {"10%": 0.75, "20%": 0.78, "30%": 0.80, "40%": 0.82, "50%": 0.84, "60%": 0.85, "70%": 0.86, "80%": 0.87, "90%": 0.88, "100%": 0.89},
                "Amazon": {"10%": 0.68, "20%": 0.71, "30%": 0.73, "40%": 0.75, "50%": 0.77, "60%": 0.78, "70%": 0.79, "80%": 0.80, "90%": 0.81, "100%": 0.83},
                "Netflix": {"10%": 0.72, "20%": 0.75, "30%": 0.77, "40%": 0.79, "50%": 0.81, "60%": 0.82, "70%": 0.83, "80%": 0.84, "90%": 0.85, "100%": 0.86}
            },
            "MatrixFactorization": {
                "MovieLens": {"10%": 0.73, "20%": 0.76, "30%": 0.78, "40%": 0.80, "50%": 0.82, "60%": 0.83, "70%": 0.84, "80%": 0.85, "90%": 0.86, "100%": 0.87},
                "Amazon": {"10%": 0.66, "20%": 0.69, "30%": 0.71, "40%": 0.73, "50%": 0.75, "60%": 0.76, "70%": 0.77, "80%": 0.78, "90%": 0.79, "100%": 0.81},
                "Netflix": {"10%": 0.70, "20%": 0.73, "30%": 0.75, "40%": 0.77, "50%": 0.79, "60%": 0.80, "70%": 0.81, "80%": 0.82, "90%": 0.83, "100%": 0.84}
            }
        },
        "RecBole": {
            "ItemKNN": {
                "MovieLens": {"10%": 0.76, "20%": 0.79, "30%": 0.81, "40%": 0.83, "50%": 0.85, "60%": 0.86, "70%": 0.87, "80%": 0.88, "90%": 0.89, "100%": 0.90},
                "Amazon": {"10%": 0.69, "20%": 0.72, "30%": 0.74, "40%": 0.76, "50%": 0.78, "60%": 0.79, "70%": 0.80, "80%": 0.81, "90%": 0.82, "100%": 0.84},
                "Netflix": {"10%": 0.73, "20%": 0.76, "30%": 0.78, "40%": 0.80, "50%": 0.82, "60%": 0.83, "70%": 0.84, "80%": 0.85, "90%": 0.86, "100%": 0.87}
            },
            "FactorMatrixization": {
                "MovieLens": {"10%": 0.74, "20%": 0.77, "30%": 0.79, "40%": 0.81, "50%": 0.83, "60%": 0.84, "70%": 0.85, "80%": 0.86, "90%": 0.87, "100%": 0.88},
                "Amazon": {"10%": 0.67, "20%": 0.70, "30%": 0.72, "40%": 0.74, "50%": 0.76, "60%": 0.77, "70%": 0.78, "80%": 0.79, "90%": 0.80, "100%": 0.82},
                "Netflix": {"10%": 0.71, "20%": 0.74, "30%": 0.76, "40%": 0.78, "50%": 0.80, "60%": 0.81, "70%": 0.82, "80%": 0.83, "90%": 0.84, "100%": 0.85}
            }
        }
    }


## 9. Run
Execute using the above function and variable definitions

In [26]:

lenskit = "LensKit"
recbole = "RecBole"
sizes = SIZES[:2]
results = prepare_results(sizes)

for dataset in Datasets:
  full = load(dataset)
  for size in sizes:
    sampled = sample(full, size)
    percentage = f"{int(size * 100)}%"

    # LensKit
    for scorer in Scorer:
      result = use_lenskit(sampled, scorer)
      results[lenskit][scorer.name][dataset.name][percentage] = result

    # RecBole
    for model in Model:
      result = use_recbole(sampled, model)
      results[recbole][model.name][dataset.name][percentage] = result

print(results)
plot_results(results)


26 Jul 11:14    INFO  creating data set from 3 x 100020 data frame
26 Jul 11:14    INFO  partitioning 100020 rows for 5970 users into 5 partitions
26 Jul 11:14    INFO  fold 0: selecting test ratings
g:\My Drive\Workspace\Thesis\Repository\unreasonable-effectiveness-recsys\venv\Lib\site-packages\lenskit\splitting\users.py:197: DataWarning: item list collection has empty lists, they will be dropped
  test_tbl = test.to_df()[["user_id", "item_id"]]
g:\My Drive\Workspace\Thesis\Repository\unreasonable-effectiveness-recsys\venv\Lib\site-packages\lenskit\data\relationships.py:378: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:55.)
  return torch.sparse_csr_tensor(
26 Jul 11:14    INFO  fold 1: selecting test ratings
g:\My

KeyboardInterrupt: 